008 DATA CLEANING

- Start with 007_data_only_pedestrians parquet file
- End with 008_data_only_pedestrians parquet file

OBJECTIVES: 
- clean 'TipoStrada' column (G)
- clean 'Visibilita' column (H)
- clean 'Tipolesione' column (I)
- clean 'Illuminazione' column (J)
- clean 'Traffico' column (K)
- clean 'TipoVeicolo' column (M)

In [1]:
import pandas as pd
import numpy as np
import pyarrow

In [2]:
df = pd.read_parquet("007_data_only_pedestrians.parquet").copy()
df.shape

(19416, 49)

The dataset currently has 51 columns and 19,416 rows of data. 

## CATEGORIES WITH AN INHERENT ORDER

In [3]:
df.columns

Index(['Protocollo', 'TipoVeicolo', 'StatoVeicolo', 'DataOraIncidente',
       'CondizioneAtmosferica', 'Traffico', 'Gruppo', 'Localizzazione1',
       'STRADA1', 'Localizzazione2', 'STRADA2', 'Strada02', 'Chilometrica',
       'DaSpecificare', 'Latitude', 'Longitude', 'TipoStrada', 'Visibilita',
       'Illuminazione', 'Tipolesione', 'Deceduto', 'DecedutoDopo',
       'parked_vehicle_involved', 'driver_injury', 'driver_deceduto',
       'driver_deceduto_dopo', 'driver_gender', 'passenger_no_of_females',
       'passenger_no_of_males', 'passenger_no_of_unknown_sex',
       'passengers_killed', 'passengers_uninjured', 'passengers_injured',
       'phase_of_day', 'YEAR', 'MONTH', 'DATE', 'TIME', 'DAY',
       'pedestrian_gender', 'road_conditions', 'road_markings_absent',
       'road_markings_onroad', 'road_markings_signposts',
       'road_markings_temporary', 'road_features',
       'road_markings_traffic_lights', 'road_surface', 'accident_type'],
      dtype='object')

In [ ]:
def analyse_column(column_name):
    """
    Analyzes value counts, percentages, and unique protocols for a column in the DataFrame.
    Permanently converts empty strings to NaN in the original DataFrame (for string-like columns).

    Args:
        column_name (str): The name of the column to analyze.

    Returns:
        pandas.DataFrame: A DataFrame with 'Count', '% per category', and 
                          'Unique Accidents' for each unique value.
    """
    # Validate input
    if column_name not in df.columns:
        raise ValueError(f"Column '{column_name}' not found in DataFrame")

    s = df[column_name]

    # Count empty strings (via temporary string view; safe for any dtype)
    empty_string_count = (s.astype("string") == "").sum()

    # If the column is object/string/categorical → convert and replace
    if (
        pd.api.types.is_object_dtype(s)
        or pd.api.types.is_string_dtype(s)
        or isinstance(s.dtype, pd.CategoricalDtype)  # modern check
    ):
        df[column_name] = s.astype("string").replace("", pd.NA)
        if empty_string_count > 0:
            print(
                f"Converted {empty_string_count} empty strings to NaN in column '{column_name}'")
    else:
        if empty_string_count > 0:
            print(
                f"Note: counted {empty_string_count} empty strings in non-string column '{column_name}', no replacement performed.")

    cleaned_column = df[column_name]

    # Calculate counts and percentages (include NaNs)
    counts = cleaned_column.value_counts(dropna=False)
    percentages = (cleaned_column.value_counts(
        dropna=False, normalize=True) * 100).round(2)

    # Calculate unique protocols for each category/value
    unique_protocols = []
    for category in counts.index:
        if pd.isna(category):
            mask = cleaned_column.isna()
        else:
            mask = cleaned_column == category
        protocol_count = df.loc[mask, 'Protocollo'].nunique()
        unique_protocols.append(protocol_count)

    # Build result
    result = pd.DataFrame({
        'Count': counts,
        '% per category': percentages,
        'Unique Accidents': unique_protocols
    }, index=counts.index)

    return result

### (G) 'TipoStrada' column

In [5]:
df['TipoStrada'] = df['TipoStrada'].replace(
    'PiÃ¹ di due carreggiate', 'Piu di due carreggiate')

In [6]:
analyse_column('TipoStrada')

,Count,% per category,Unique Accidents
TipoStrada,,,
Una carreggiata a doppio senso,9986,51.43,9190
Una carreggiata a senso unico di marcia,4543,23.4,4216
Due carreggiate,3601,18.55,3334
Piu di due carreggiate,1238,6.38,1170
Una carreggiata a senso unico alternato,44,0.23,40
<NA>,4,0.02,4


To code the 'TipoStrada' data, we introduce a 'level of difficulty' for the pedestrian:

- One lane - look one way	1
- One lane - look two ways	2
- Two lanes	3
- More than two lanes	4

This corresponds to the following coding:

- 'Due carreggiate'	3
- 'Piu di due carreggiate'	4
- 'Una carreggiata a doppio senso'	2
- 'Una carreggiata a senso unico alternato'	2
- 'Una carreggiata a senso unico di marcia'	1

In [7]:
df['TipoStradaDifficulty'] = df['TipoStrada'].map({
    'Una carreggiata a senso unico di marcia': 1,
    'Una carreggiata a doppio senso': 2,
    'Una carreggiata a senso unico alternato': 2,
    'Due carreggiate': 3,
    'Piu di due carreggiate': 4,
    '': np.nan                                 # using NaN for unknown
})

We are missing information about 4 locations; given the small number, we can try to look them up on Google Maps, using the terrain / satellite view to visually identity the number of lanes. 

In [8]:
# Get all rows where TipoStradaDifficulty is missing
na_mask = df["TipoStradaDifficulty"].isna()

# Select columns + reset index for cleaner display
result = (
    df.loc[na_mask, ["STRADA1", "STRADA2", "Chilometrica",
                     "DaSpecificare", "Latitude", "Longitude"]]
    .reset_index()   # keep original row number in a column
)

# Display
print(result.to_string(index=False))

 index             STRADA1          STRADA2 Chilometrica                               DaSpecificare  Latitude  Longitude
 15300    PIAZZA DI SPAGNA VIA DEI CONDOTTI                                                                NaN        NaN
 15633  VIA FILIPPO TURATI                           129                                               41.8972    12.5046
 17647        PONTE CAVOUR                               attraversamento pedonale lato via tomacelli       NaN        NaN
 18696 PIAZZA DEL COLOSSEO                                                        arco di costantino       NaN        NaN


Looking up these values on Google maps, we see:
- PIAZZA DI SPAGNA VIA DEI CONDOTTI: Due carreggiate (3)
- VIA FILIPPO TURATI 129: Due carreggiate (3)
- PONTE CAVOUR attraversamento pedonale lato via tomacelli: Piu di due carreggiate (4)
- PIAZZA DEL COLOSSEO arco di costantino: Piu di due carreggiate (4)

In [9]:
# Set values using .loc with .index
indices = df.loc[na_mask].index

# First two rows should be 3, next two rows should be 4
df.loc[indices[:2], "TipoStradaDifficulty"] = 3
df.loc[indices[2:], "TipoStradaDifficulty"] = 4

In [10]:
analyse_column('TipoStradaDifficulty')

,Count,% per category,Unique Accidents
TipoStradaDifficulty,,,
2.0,10030,51.66,9230
1.0,4543,23.40,4216
3.0,3603,18.56,3336
4.0,1240,6.39,1172


In [11]:
df['TipoStradaDifficulty'].dtype

dtype('float64')

In [12]:
df["TipoStradaDifficulty"] = pd.Categorical(
    df["TipoStradaDifficulty"],
    categories=[1, 2, 3, 4],  # define the order
    ordered=True
)

#### (G) 'TipoStrada' column - ENCODING COMPLETE
- 1 low difficulty
- 4 high difficulty

### (H) 'Visibilita' column

In [13]:
analyse_column('Visibilita')

,Count,% per category,Unique Accidents
Visibilita,,,
Buona,15104,77.79,14001
Sufficiente,3669,18.9,3370
Insufficiente,641,3.3,581
<NA>,2,0.01,2


In this category, we will have ordered categories. We will fill in the mode for the missing values. 

In [14]:
df['Visibilita'] = df['Visibilita'].fillna(df['Visibilita'].mode()[0])

In [15]:
df['visibility'] = df['Visibilita'].map({
    'Buona': 2,
    'Sufficiente': 1,
    'Insufficiente': 0
})
# Drop the original 'Visibilita' column
df = df.drop('Visibilita', axis=1)
df['visibility'].value_counts(dropna=False)

visibility
2    15106
1     3669
0      641
Name: count, dtype: int64

In [16]:
df["visibility"] = pd.Categorical(
    df["visibility"],
    categories=[0, 1, 2],  # define the order
    ordered=True
)

#### (H) 'Visibilita' column - ENCODING COMPLETE
- 0 insufficient visibility
- 2 good visibility

### (I) 'Tipolesione' column

DATA CODING: Tipolesione

With a little help from a policeperson who teaches other police how to collect information from accidents, we put the type of pedestrian injury in order, with 0 being the least injured and 3 and 4 being fatal:

- Illeso = 0
- Rifiuta cure immediate = 0
- Rimandato = 1
- Ricoverato = 1
- Prognosi riservata = 2
- Deceduto dopo = 3
- Deceduto durante prime cure / Deceduto durante trasporto / Deceduto sul posto = 4

In [17]:
analyse_column('Tipolesione')

,Count,% per category,Unique Accidents
Tipolesione,,,
Rimandato,13273,68.36,12531
Ricoverato,2541,13.09,2486
Illeso,1663,8.57,1559
Rifiuta cure immediate,968,4.99,926
Prognosi riservata,725,3.73,718
Deceduto durante prime cure,124,0.64,123
Deceduto sul posto,99,0.51,99
Deceduto durante trasporto,23,0.12,23


In [18]:
# Define the mapping
pedestrian_injury_mapping = {
    'Illeso': 'No_injury',
    'Rifiuta cure immediate': 'No_injury',
    'Rimandato': 'Some',
    'Ricoverato': 'Some',
    'Prognosi riservata': 'Severe',
    'Deceduto durante prime cure': 'Died_immediately',
    'Deceduto sul posto': 'Died_immediately',
    'Deceduto durante trasporto': 'Died_immediately'
}

# Create the new Injury column
df['Injury'] = df['Tipolesione'].map(pedestrian_injury_mapping)

NB No null values. 

In [19]:
analyse_column('Injury')

,Count,% per category,Unique Accidents
Injury,,,
Some,15814,81.45,14896
No_injury,2631,13.55,2463
Severe,725,3.73,718
Died_immediately,246,1.27,244


In [20]:
analyse_column('Deceduto')

,Count,% per category,Unique Accidents
Deceduto,,,
0,19170,98.73,17730
-1,246,1.27,244


This seems to correspond with the 246 pedestrians who died immediately. We will check:

In [21]:
# Filter rows where Injury is 'Died_immediately'
died_immediately_rows = df[df['Injury'] == 'Died_immediately']

# Check if all Deceduto values are -1
all_match = (died_immediately_rows['Deceduto'] == -1).all()

print("Do all 'Died_immediately' rows have Deceduto = -1?", all_match)

# Optionally, check which rows don't match
mismatch = died_immediately_rows[died_immediately_rows['Deceduto'] != -1]
print("Rows where Injury='Died_immediately' but Deceduto != -1:",
      mismatch.shape[0])

Do all 'Died_immediately' rows have Deceduto = -1? True
Rows where Injury='Died_immediately' but Deceduto != -1: 0


In [22]:
analyse_column('DecedutoDopo')

Converted 1909 empty strings to NaN in column 'DecedutoDopo'


,Count,% per category,Unique Accidents
DecedutoDopo,,,
NON DECEDUTO,17306,89.13,16210
<NA>,1909,9.83,1802
DECEDUTO ENTRO 2 MESI,41,0.21,41
DECEDUTO ENTRO 15 GIORNI,33,0.17,33
DECEDUTO ENTRO 1 MESE,24,0.12,24
DECEDUTO DA 25 A 48 ORE DOPO,22,0.11,22
DECEDUTO DA 13 A 24 ORE DOPO,14,0.07,14
DECEDUTO ENTRO LE DODICI ORE,12,0.06,12
DECEDUTO ENTRO 3 MESI,10,0.05,10


Now we lump together all categories where the pedestrian died into 'Died_later' (those who died on the spot are identified in the separate feature: Injury)

In [23]:
df['Survival'] = np.where(
    df['DecedutoDopo'].isna(), 'Unknown',
    np.where(df['DecedutoDopo'].str.contains('NON', na=False), 'Survived',
             np.where(df['DecedutoDopo'].str.contains('ENTRO|DA', na=False), 'Died_later', df['DecedutoDopo']))
)

In [24]:
analyse_column('Survival')

,Count,% per category,Unique Accidents
Survival,,,
Survived,17306,89.13,16210
Unknown,1909,9.83,1802
Died_later,201,1.04,201


In [25]:
table_survival = pd.crosstab(df['Survival'], df['Injury'])
print(table_survival)

Injury      Died_immediately  No_injury  Severe   Some
Survival                                              
Died_later                 0          0     136     65
Survived                   0        968     589  15749
Unknown                  246       1663       0      0


- Everyone in the row 'Survived' is in the correct place. 
- Everyone in the 'Died_later' row needs Injury: Died_later

The 1721 people who did not have an ambulance called or the ambulance was called but was sent away were most likely uninjured. 
Examining the table above, of the people with some injury (15824) and thus WERE taken to hospital, only 65 died. This is 0.4%. 
Therefore it is highly unlikely that anyone deemed uninjured (ambulance called but did not take them to hospital) or did not want an ambulance called at all died afterwards. Therefore, we will also move the 1721 'Unknown' survival into the 'Survived' category.

In [26]:
# Introduce 'Died_later' category to Injury column:
df.loc[df['Survival'] == 'Died_later', 'Injury'] = 'Died_later'

In [27]:
# If there was no injury, they survived:
df.loc[df['Injury'] == 'No_injury', 'Survival'] = 'Survived'

In [28]:
# Introduce 'Died_immediately' category to Survival column:
df.loc[df['Survival'] == 'Unknown', 'Survival'] = 'Died_immediately'

In [29]:
table_survival = pd.crosstab(df['Survival'], df['Injury'])
print(table_survival)

Injury            Died_immediately  Died_later  No_injury  Severe   Some
Survival                                                                
Died_immediately               246           0          0       0      0
Died_later                       0         201          0       0      0
Survived                         0           0       2631     589  15749


In [30]:
df['Injury'] = df['Injury'].map({
    'Died_immediately': 4,
    'Died_later': 3,
    'Severe': 2,
    'Some': 1,
    'No_injury': 0
})
# Drop the original 'Deceduto'. 'DecedutoDopo' and 'Tipolesione' columns
df = df.drop(['DecedutoDopo', 'Deceduto', 'Tipolesione'], axis=1)
analyse_column('Injury')

,Count,% per category,Unique Accidents
Injury,,,
1,15749,81.11,14834
0,2631,13.55,2463
2,589,3.03,584
4,246,1.27,244
3,201,1.04,201


Now we will repeat the same process for the driver:      

In [31]:
analyse_column('driver_injury')

,Count,% per category,Unique Accidents
driver_injury,,,
Illeso,16879,86.93,15630
Rimandato,1112,5.73,997
<NA>,951,4.9,894
Rifiuta cure immediate,313,1.61,289
Ricoverato,132,0.68,121
Prognosi riservata,23,0.12,18
Deceduto sul posto,5,0.03,4
Deceduto durante trasporto,1,0.01,1


As we have some 'NAN' (None) values in this case, let's first make sure it is clear these refer to 'unknown' and not no injury:

In [32]:
df['driver_injury'] = df['driver_injury'].replace(np.nan, 'Unknown')

In [33]:
analyse_column('driver_injury')

,Count,% per category,Unique Accidents
driver_injury,,,
Illeso,16879,86.93,15630
Rimandato,1112,5.73,997
Unknown,951,4.9,894
Rifiuta cure immediate,313,1.61,289
Ricoverato,132,0.68,121
Prognosi riservata,23,0.12,18
Deceduto sul posto,5,0.03,4
Deceduto durante trasporto,1,0.01,1


In [34]:
# Define the mapping
driver_injury_mapping = {
    'Illeso': 'No_injury',
    'Rifiuta cure immediate': 'No_injury',
    'Rimandato': 'Some',
    'Ricoverato': 'Some',
    'Prognosi riservata': 'Severe',
    'Deceduto durante prime cure': 'Died_immediately',
    'Deceduto sul posto': 'Died_immediately',
    'Deceduto durante trasporto': 'Died_immediately',
    'Unknown': 'Unknown'
}

# Create the new values in the driver_injury column, using the same mapping as before


df['driver_injury'] = df['driver_injury'].map(driver_injury_mapping)

In [35]:
analyse_column('driver_injury')

,Count,% per category,Unique Accidents
driver_injury,,,
No_injury,17192,88.55,15919
Some,1244,6.41,1118
Unknown,951,4.9,894
Severe,23,0.12,18
Died_immediately,6,0.03,5


In [36]:
analyse_column('driver_deceduto')

,Count,% per category,Unique Accidents
driver_deceduto,,,
0,18459,95.07,17055
<NA>,951,4.9,894
-1,6,0.03,5


The '-1' row seems to correspond with the 6 drivers who died immediately. We will check:

In [37]:
# Filter rows where Injury is 'Died_immediately'
driver_died_immediately_rows = df[df['driver_injury'] == 'Died_immediately']

# Check if all Deceduto values are -1
all_match = (driver_died_immediately_rows['driver_deceduto'] == -1).all()

print("Do all 'Died_immediately' rows have Deceduto = -1?", all_match)

# Optionally, check which rows don't match
mismatch = driver_died_immediately_rows[driver_died_immediately_rows['driver_deceduto'] != -1]
print("Rows where Injury='Died_immediately' but Deceduto != -1:",
      mismatch.shape[0])

Do all 'Died_immediately' rows have Deceduto = -1? True
Rows where Injury='Died_immediately' but Deceduto != -1: 0


In [38]:
analyse_column('driver_deceduto_dopo')

,Count,% per category,Unique Accidents
driver_deceduto_dopo,,,
<NA>,17836,91.86,16529
NON DECEDUTO,1577,8.12,1423
DECEDUTO ENTRO IL TERZO GIORNO,2,0.01,1
DECEDUTO ENTRO 15 GIORNI,1,0.01,1


Now we lump together all categories where the driver died into 'Died_later' (those who died on the spot are identified in the separate feature: driver_injury)

In [39]:
df['driver_survival'] = np.where(
    df['driver_deceduto_dopo'].isna(), 'Unknown',
    np.where(df['driver_deceduto_dopo'].str.contains('NON', na=False), 'Survived',
             np.where(df['driver_deceduto_dopo'].str.contains('ENTRO|DA', na=False), 'Died_later', df['driver_deceduto_dopo']))
)

In [40]:
analyse_column('driver_survival')

,Count,% per category,Unique Accidents
driver_survival,,,
Unknown,17836,91.86,16529
Survived,1577,8.12,1423
Died_later,3,0.02,2


In [41]:
table_driver_survival = pd.crosstab(df['driver_survival'].fillna(
    'Unknown'), df['driver_injury'].fillna('Unknown'))
print(table_driver_survival)

driver_injury    Died_immediately  No_injury  Severe  Some  Unknown
driver_survival                                                    
Died_later                      0          0       2     1        0
Survived                        0        313      21  1243        0
Unknown                         6      16879       0     0      951


- Everyone in the row 'Survived' is in the correct place. 
- Everyone in the 'Died_later' row needs Injury: Died_later

The 16879 people who did not have an ambulance called or the ambulance was called but was sent away were most likely uninjured. 
Examining the table above, of the drivers with some or severe injury (1264) and thus WERE taken to hospital, none died. Therefore it is highly unlikely that any driver deemed uninjured (ambulance called but did not take them to hospital) or did not want an ambulance called at all died afterwards. Therefore, we will also move the 16879 'Unknown' survival into the 'Survived' category.

In [42]:
# Introduce 'Died_later' category to Injury column:
df.loc[df['driver_survival'] == 'Died_later', 'driver_injury'] = 'Died_later'

In [43]:
# If there was no injury, they survived:
df.loc[df['driver_injury'] == 'No_injury', 'driver_survival'] = 'Survived'

In [44]:
# Introduce 'Died_immediately' category to Survival column:
df.loc[df['driver_injury'] == 'Died_immediately',
       'driver_survival'] = 'Died_immediately'

In [45]:
table_driver_survival = pd.crosstab(df['driver_survival'].fillna(
    'Unknown'), df['driver_injury'].fillna('Unknown'))
print(table_driver_survival)

driver_injury     Died_immediately  Died_later  No_injury  Severe  Some  \
driver_survival                                                           
Died_immediately                 6           0          0       0     0   
Died_later                       0           3          0       0     0   
Survived                         0           0      17192      21  1243   
Unknown                          0           0          0       0     0   

driver_injury     Unknown  
driver_survival            
Died_immediately        0  
Died_later              0  
Survived                0  
Unknown               951  


To try to better understand the dynamics of the accidents where the driver_injury is unknown and the driver_survival is unknown, we can look at the state of the vehicle:

In [47]:
df[df['driver_injury'] == 'Unknown']['StatoVeicolo'].value_counts(dropna=False)

StatoVeicolo
Allontanatosi                    868
In marcia / fermata / arresto     83
                                   0
Sosta                              0
Name: count, dtype: int64

In 91% of these cases (868), the vehicle did not stop. We will assume that if the driver was able to drive away, they were uninjured. If statoveicolo is 'Allontanatosi' and driver_injury is Unknown, we assume the driver_injury was No_injury. 

In [48]:
df.loc[(df['StatoVeicolo'] == 'Allontanatosi')
       & ((df['driver_injury'] == 'Unknown') | df['driver_injury'].isna()),
       'driver_injury'] = 'No_injury'

df.loc[(df['StatoVeicolo'] == 'Allontanatosi')
       & ((df['driver_survival'] == 'Unknown') | df['driver_survival'].isna()),
       'driver_survival'] = 'Survived'

We examine the 83 accidents where the driver stopped and see how severe the accident was:

In [49]:
df[(df['driver_injury'] == 'Unknown') &
   (df['StatoVeicolo'] == 'In marcia / fermata / arresto')]['Injury'].value_counts()

Injury
1    69
0    13
4     1
Name: count, dtype: int64

In [50]:
df[(df['driver_injury'] == 'Unknown') &
   (df['StatoVeicolo'] == 'In marcia / fermata / arresto') &
   (df['Injury'] == 4)]

,Protocollo,TipoVeicolo,StatoVeicolo,DataOraIncidente,CondizioneAtmosferica,Traffico,Gruppo,Localizzazione1,STRADA1,Localizzazione2,...,road_markings_temporary,road_features,road_markings_traffic_lights,road_surface,accident_type,TipoStradaDifficulty,visibility,Injury,Survival,driver_survival
12509,4166272,Autovettura privata,In marcia / fermata / arresto,2019-01-07 04:55:00+01:00,Sereno,Scarso,3,Strada Urbana,CORSO D'ITALIA,da specificare,...,0,Straight road,0,Tarmac,accident_pedestrian_run_over,1,0,4,Died_immediately,Unknown


In this case, the pedestrian did not survive. The vehicle was a car. Given that in most cases where a car is involved, the driver is uninjured, we will assume this driver is also uninjured. 

In [51]:
mask = ((df['driver_injury'] == 'Unknown') | (df['driver_injury'].isna()))
df.loc[mask, 'driver_survival'] = 'Survived'
df.loc[mask, 'driver_injury'] = 'No_injury'

In [52]:
table_driver_survival = pd.crosstab(df['driver_survival'].fillna(
    'Unknown'), df['driver_injury'].fillna('Unknown'))
print(table_driver_survival)

driver_injury     Died_immediately  Died_later  No_injury  Severe  Some
driver_survival                                                        
Died_immediately                 6           0          0       0     0
Died_later                       0           3          0       0     0
Survived                         0           0      18143      21  1243


In [53]:
analyse_column('driver_injury')

,Count,% per category,Unique Accidents
driver_injury,,,
No_injury,18143,93.44,16813
Some,1243,6.4,1117
Severe,21,0.11,17
Died_immediately,6,0.03,5
Died_later,3,0.02,2


In [54]:
df['driver_injury'] = df['driver_injury'].map({
    'Died_immediately': 4,
    'Died_later': 3,
    'Severe': 2,
    'Some': 1,
    'No_injury': 0
})

analyse_column('driver_injury')

,Count,% per category,Unique Accidents
driver_injury,,,
0,18143,93.44,16813
1,1243,6.40,1117
2,21,0.11,17
4,6,0.03,5
3,3,0.02,2


In [ ]:
# Drop the original 'Deceduto'. 'DecedutoDopo' and 'Tipolesione' columns for the driver:
df = df.drop(['driver_deceduto', 'driver_deceduto_dopo',
             'driver_survival'], axis=1)

Change name of 'Injury' column to 'pedestrian_injury' & convert these features to dtype ordered categorical

In [56]:
df["pedestrian_injury"] = pd.Categorical(
    df["Injury"],
    categories=[0, 1, 2, 3, 4],  # define the order
    ordered=True
)
df = df.drop(columns="Injury")

df["driver_injury"] = pd.Categorical(
    df["driver_injury"],
    categories=[0, 1, 2, 3, 4],  # define the order
    ordered=True
)

In [57]:
analyse_column('pedestrian_injury')

,Count,% per category,Unique Accidents
pedestrian_injury,,,
1,15749,81.11,14834
0,2631,13.55,2463
2,589,3.03,584
4,246,1.27,244
3,201,1.04,201


In [58]:
analyse_column('driver_injury')

,Count,% per category,Unique Accidents
driver_injury,,,
0,18143,93.44,16813
1,1243,6.4,1117
2,21,0.11,17
4,6,0.03,5
3,3,0.02,2


#### (I) 'Tipolesione' column + 'driver_injury' column ENCODING COMPLETE

CATEGORICAL COLUMNS: pedestrian_injury and driver_injury

- 'Died_immediately': 4,
- 'Died_later': 3,
- 'Severe': 2,
- 'Some': 1,
- 'No_injury': 0

### (J) 'Illuminazione' column

In [59]:
analyse_column('Illuminazione')

,Count,% per category,Unique Accidents
Illuminazione,,,
Ore Diurne,10483,53.99,9755
<NA>,4179,21.52,3859
Sufficiente,4170,21.48,3803
Insufficiente,486,2.5,444
Inesistente,98,0.5,93


To compare the 'Illuminazione' values logged with the times of day identified (daylight, twilight, nighttime, etc), we can make a table to see illumination and phase of day:

In [60]:
illumination_table = pd.crosstab(df['phase_of_day'].fillna(
    'Unknown'), df['Illuminazione'].fillna('Unknown'))
print(illumination_table)

Illuminazione                  Inesistente  Insufficiente  Ore Diurne  \
phase_of_day                                                            
time_astronomical_twilight_am            3              5           7   
time_astronomical_twilight_pm           16             81          19   
time_civil_twilight_am                   0              7          99   
time_civil_twilight_pm                   6             43          83   
time_daylight_am                         0              1        5485   
time_daylight_pm                         3             17        4675   
time_nautical_twilight_am                4             10          39   
time_nautical_twilight_pm               18             79          36   
time_night_am                           12             39           3   
time_night_pm                           36            204          37   

Illuminazione                  Sufficiente  Unknown  
phase_of_day                                         
time_astronomic

Given: the large amount of values in the 'unknown' column (over 4000, around 21%); the selection of 'ore_diurne' where the hours are late in the evening, during the night, or very early in the morning (over 100) it appears this data has not been correctly collected and is open to personal interpretation. It is difficult to determine the criteria used by data collectors to differentiate between 'Ore Diurne' and 'Sufficiente' for daytime accidents. Therefore we will transform this into a binary column where only the lack of adequate lighting is noted. 

In [61]:
df['lighting_insufficient'] = np.where(
    df['Illuminazione'].isin(['Insufficiente', 'Inesistente']), 1, 0)

In [62]:
analyse_column('lighting_insufficient')

,Count,% per category,Unique Accidents
lighting_insufficient,,,
0,18832,96.99,17417
1,584,3.01,537


In [63]:
df["lighting_insufficient"] = pd.Categorical(
    df["lighting_insufficient"],
    categories=[0, 1],  # define the order
    ordered=False
)

#### (J) 'Illuminazione' column - ENCODING COMPLETE: 
- lighting_insufficient 0 or 1

### (K) TRAFFICO COLUMN

In [64]:
analyse_column('Traffico')

,Count,% per category,Unique Accidents
Traffico,,,
Normale,12940,66.65,11971
Scarso,4611,23.75,4244
Intenso,1846,9.51,1721
<NA>,19,0.1,18


Here we try to find other accidents within 2 hours in the same area of Rome (same reporting group of police: Gruppo):

In [65]:
protocols_no_traffico_density = df[df['Traffico'].isna(
)]['Protocollo'].unique().tolist()

missing_traffic_count = df['Traffico'].isna().sum()
print(f"Starting with {missing_traffic_count} missing Traffico values")

updated_protocols = []
no_match_protocols = []

for i, protocol in enumerate(protocols_no_traffico_density):
    if i % 100 == 0:
        print(
            f'Processing protocol {i+1} / {len(protocols_no_traffico_density)}')

    # Get the protocol, reporting group and accident date and time for the accident in question:
    target_accident = df[df['Protocollo'] == protocol].iloc[0]
    target_gruppo = target_accident['Gruppo']
    target_time = target_accident['DataOraIncidente']

    # Set time difference to 1 hour from the original accident
    time_window = pd.Timedelta(hours=2)

    # Find other accidents reported by the same group (same area) within 1 hour that have a 'Traffico' value:
    potential_matches = df[(df['Gruppo'] == target_gruppo) &
                           (df['Protocollo'] != protocol) &
                           (abs(df['DataOraIncidente'] - target_time) <= time_window) &
                           (df['Traffico'].notna())]

    # Debug: Check how many potential matches we found
    if len(potential_matches) == 0:
        no_match_protocols.append(protocol)
        continue

    # Prevent repetition of rows with the same protocol (i.e. the same accident)
    similar_accidents = potential_matches.groupby(
        'Protocollo').first().reset_index()

    if len(similar_accidents) > 0:
        # Count how many people (rows) will be updated for this protocol
        people_in_accident = len(df[df['Protocollo'] == protocol])

        # Use the most common Traffico value from matching accidents
        traffic_values = similar_accidents['Traffico'].value_counts()
        most_common_traffic = traffic_values.index[0]

        # Update the original accident (all people involved)
        df.loc[df['Protocollo'] == protocol, 'Traffico'] = most_common_traffic
        updated_protocols.append(protocol)

        print(f"Protocol {protocol}: Updated Traffico to '{most_common_traffic}' "
              f"for {people_in_accident} people based on {len(similar_accidents)} matching accidents")

# Check results
final_missing_count = df['Traffico'].isna().sum()
actually_updated = missing_traffic_count - final_missing_count

print(f"\nSummary:")
print(f"Original missing Traffico values: {missing_traffic_count}")
print(f"Protocols with no matches: {len(no_match_protocols)}")
print(f"Protocols (accidents) updated: {len(updated_protocols)}")
print(f"People (rows) actually updated: {actually_updated}")
print(f"Remaining missing: {final_missing_count}")
print(f"Success rate: {len(updated_protocols)}/{len(protocols_no_traffico_density)} protocols ({len(updated_protocols)/len(protocols_no_traffico_density)*100:.1f}%)")

Starting with 19 missing Traffico values
Processing protocol 1 / 18
Protocol 2416585: Updated Traffico to 'Normale' for 1 people based on 1 matching accidents
Protocol 3380172: Updated Traffico to 'Normale' for 1 people based on 1 matching accidents
Protocol 5187106: Updated Traffico to 'Normale' for 2 people based on 1 matching accidents
Protocol 5254295: Updated Traffico to 'Normale' for 1 people based on 1 matching accidents

Summary:
Original missing Traffico values: 19
Protocols with no matches: 14
Protocols (accidents) updated: 4
People (rows) actually updated: 5
Remaining missing: 14
Success rate: 4/18 protocols (22.2%)


Here we try to look for accidents within 1 hour of the original accident but on the same day of the week 1 or 2 weeks before or after the original accident:

In [66]:
protocols_no_traffico_density = df[df['Traffico'].isna(
)]['Protocollo'].unique().tolist()

missing_traffic_count = df['Traffico'].isna().sum()
print(f"Starting with {missing_traffic_count} missing Traffico values")

updated_protocols = []
no_match_protocols = []

for i, protocol in enumerate(protocols_no_traffico_density):
    if i % 100 == 0:
        print(
            f'Processing protocol {i+1} / {len(protocols_no_traffico_density)}')

    # Get the protocol, reporting group and accident date and time for the accident in question:
    target_accident = df[df['Protocollo'] == protocol].iloc[0]
    target_gruppo = target_accident['Gruppo']
    target_time = target_accident['DataOraIncidente']

    # Calculate the 4 target dates: same day of week, ±1 and ±2 weeks
    one_week_before = target_time - pd.Timedelta(weeks=1)
    two_weeks_before = target_time - pd.Timedelta(weeks=2)
    one_week_after = target_time + pd.Timedelta(weeks=1)
    two_weeks_after = target_time + pd.Timedelta(weeks=2)

    target_dates = [one_week_before, two_weeks_before,
                    one_week_after, two_weeks_after]

    # Find other accidents reported by the same group on target dates within ±1 hour
    potential_matches = df[
        (df['Gruppo'] == target_gruppo) &
        (df['Protocollo'] != protocol) &
        # Check if accident date matches any of our target dates (same day)
        (df['DataOraIncidente'].dt.date.isin([d.date() for d in target_dates])) &
        # Within ±1 hour of the original accident time
        (abs(df['DataOraIncidente'].dt.hour - target_time.hour) <= 1.5) &
        (df['Traffico'].notna())
    ]

    # Debug: Check how many potential matches we found
    if len(potential_matches) == 0:
        no_match_protocols.append(protocol)
        continue

    # Prevent repetition of rows with the same protocol (i.e. the same accident)
    similar_accidents = potential_matches.groupby(
        'Protocollo').first().reset_index()

    if len(similar_accidents) > 0:
        # Count how many people (rows) will be updated for this protocol
        people_in_accident = len(df[df['Protocollo'] == protocol])

        # Use the most common Traffico value from matching accidents
        traffic_values = similar_accidents['Traffico'].value_counts()
        most_common_traffic = traffic_values.index[0]

        # Update the original accident (all people involved)
        df.loc[df['Protocollo'] == protocol, 'Traffico'] = most_common_traffic
        updated_protocols.append(protocol)

        print(f"Protocol {protocol}: Updated Traffico to '{most_common_traffic}' "
              f"for {people_in_accident} people based on {len(similar_accidents)} matching accidents")

# Check results
final_missing_count = df['Traffico'].isna().sum()
actually_updated = missing_traffic_count - final_missing_count

print(f"\nSummary:")
print(f"Original missing Traffico values: {missing_traffic_count}")
print(f"Protocols with no matches: {len(no_match_protocols)}")
print(f"Protocols (accidents) updated: {len(updated_protocols)}")
print(f"People (rows) actually updated: {actually_updated}")
print(f"Remaining missing: {final_missing_count}")

if len(protocols_no_traffico_density) > 0:
    success_rate = len(updated_protocols) / \
        len(protocols_no_traffico_density)*100
    print(
        f"Success rate: {len(updated_protocols)}/{len(protocols_no_traffico_density)} protocols ({success_rate:.1f}%)")
else:
    print("No protocols needed updating!")

Starting with 14 missing Traffico values
Processing protocol 1 / 14
Protocol 4142485: Updated Traffico to 'Normale' for 1 people based on 1 matching accidents
Protocol 5900691: Updated Traffico to 'Normale' for 1 people based on 1 matching accidents

Summary:
Original missing Traffico values: 14
Protocols with no matches: 12
Protocols (accidents) updated: 2
People (rows) actually updated: 2
Remaining missing: 12
Success rate: 2/14 protocols (14.3%)


Now we proceed with the traffic_density coding:
- intense traffic = 2
- normal traffic = 1
- little traffic = 0
- Nan = the mode

In [67]:
df['traffic_density'] = df['Traffico'].map({
    'Intenso': 2,
    'Normale': 1,
    'Scarso': 0,
})

# Fill NA with the mode of the mapped values
mode_val = df['traffic_density'].mode()[0]
df['traffic_density'] = df['traffic_density'].fillna(mode_val)

In [68]:
df["traffic_density"] = pd.Categorical(
    df["traffic_density"],
    categories=[0, 1, 2],  # define the order
    ordered=True
)

In [69]:
analyse_column('traffic_density')

,Count,% per category,Unique Accidents
traffic_density,,,
1,12959,66.74,11989
0,4611,23.75,4244
2,1846,9.51,1721


#### (K) TRAFFICO COLUMN - ENCODING COMPLETE

### (M) TIPOVEICOLO COLUMN

In [70]:
analyse_column('TipoVeicolo')

,Count,% per category,Unique Accidents
TipoVeicolo,,,
Autovettura privata,12731,65.57,11682
Motociclo a solo,3354,17.27,3137
Autocarro inferiore 35 q.,852,4.39,802
Unknown,612,3.15,578
Autovettura pubblica,516,2.66,492
Motociclo con passeggero,373,1.92,338
Ciclomotore,196,1.01,184
Autobus di linea,148,0.76,141
Autobus urbano,98,0.5,97


In [71]:
df['TipoVeicolo'].nunique()

41

In [72]:
df['TipoVeicolo'].unique()

<StringArray>
[               'Autocarro inferiore 35 q.',
                     'Autovettura pubblica',
                                  'Unknown',
                      'Autovettura privata',
                         'Motociclo a solo',
                       'Vettura tranviaria',
                 'Motociclo con passeggero',
                              'Ciclomotore',
                         'Autobus di linea',
                           'Autobus urbano',
                   'Autovettura di polizia',
                'Autocarro superiore 35 q.',
                      'Quadriciclo leggero',
                               'Velocipede',
                                'Motocarro',
               'Autoveicolo trasp.promisc.',
               'Ciclomotore con passeggero',
                         'Veicolo speciale',
                              'Autocaravan',
                                    'Altro',
                        'Autobus turistico',
                      'Macchina operatric

Given there are 41 categories, we should try to group these by weight:

🚶‍♂️ 1. Pedal-powered & Ultra-light (Least Harm)
- Velocipede
- Veicolo a braccia
- Veicolo a trazione animale

Human-powered—minimal mass.

🛴 2. Micro‑mobility (Electric Scooters & Hover)
- Monopattino elettrico
- Micromobilità elettrica - Segway / Hoverboard / Monowheel

Lightweight electric personal transport (<100 kg), moderate risk.

⚙️ 3. Light Motorised Two‑&‑Three‑Wheelers
- Bicicletta elettrica
- Ciclomotore (+ con passeggero)
- Motociclo a solo / con passeggero
- Motociclo di polizia
- Filobus (trolleybus fits here by mass)

(~100–300 kg), higher speed than micro-mobility but still far lighter than cars.

🚗 4. Cars & Authority Vehicles
- Autovettura privata / pubblica
- Autovettura di polizia / soccorso
- Autovettura con rimorchio

(1–3 tonne), moderate pedestrian risk.

🚐 5. Light Commercial Vehicles & Campers
- Autocaravan
- Autocarro inferiore 35 q. (~3.5 t)
- Motocarro / Motocarrozzetta

(~2–4 t), larger frontal area—more impact severity.

🛻 6. Medium‑Duty Trucks & Vans
- Autocarro superiore 35 q.
- Autotreno
- Autogru
- Autopompa (fire truck)
- Autosnodato
- Macchina operatrice
- Rimorchio

(~4–15 t), significant impact mass and rigidity.

🚍 7. Buses & Coaches
- Autobus urbano / di linea / turistico / extraurbana
- Vettura tranviaria

(~10–20 t), high front-end leading to torso/head impact risk.)

🚓 8. Heavy Commercial & Special Vehicles
- Autopompa (overlap)
- Veicolo speciale
- Motrice
- Autoveicolo transpromisc (e.g. police carrier)

(~15–30 t), heavy rigid vehicles with high mass and stiffness.)

🚜 9. Agricultural & Construction Machines
- Trattore agricolo / stradale
- Veicolo speciale (e.g., road grader)

(~5–30 t), high ground clearance—serious pedestrian danger.)

🚂 10. Trains & Trams
- Treno
- Vettura tranviaria (tram)

(Very heavy, steel rails—most hazardous.)

📌 Overview Table
- Group	Vehicle Classes	Examples
- 1	Pedal-powered	Velocipede, animal-pulled
- 2	Micro-mobility	Electric scooters, hoverboards
- 3	Light motorised	E‑bikes, mopeds, police motorcycles
- 4	Cars	Private, police, rescue vehicles
- 5	Light commercial	Campers, small trucks
- 6	Medium-duty	Trucks, cranes, trailers
- 7	Buses/coaches	Urban & tourist buses, trams
- 8	Heavy commercial	Special vehicles, heavy trucks
- 9	Farm & construction	Tractors, road machines
- 10	Rail vehicles	Trains, trams

https://pmc.ncbi.nlm.nih.gov/articles/PMC3217548/
Child and Adult Pedestrian Impact: The Influence of Vehicle Type on Injury Severity

In [73]:
df[(df['TipoVeicolo'] == 'Unknown') &
    (df['StatoVeicolo'] != 'Allontanatosi')]

,Protocollo,TipoVeicolo,StatoVeicolo,DataOraIncidente,CondizioneAtmosferica,Traffico,Gruppo,Localizzazione1,STRADA1,Localizzazione2,...,road_features,road_markings_traffic_lights,road_surface,accident_type,TipoStradaDifficulty,visibility,Survival,pedestrian_injury,lighting_insufficient,traffic_density


As every accident with a missing vehicle is a hit and run (StatoVeicolo: Allontanatosi and TipoVeicolo: Unknown), we shall keep these in a separate category.

First we clean 'MicromobilitÃ\xa0 elettrica - Hoverboard' ane replace it with 'Micromobilita elettrica- Hoverboard':

In [74]:
df['TipoVeicolo'] = df['TipoVeicolo'].apply(
    lambda x: 'Micromobilita elettrica - Hoverboard' if isinstance(
        x, str) and 'elettrica - Hoverboard' in x else x
)

In [75]:
df['vehicle_type'] = df['TipoVeicolo'].map({
    # Group 1 – Pedal-powered & ultra-light
    'Monopattino elettrico': 1,
    'Velocipede': 1,
    'Micromobilita elettrica - Hoverboard': 1,

    # Group 2 – Micro-mobility (electric scooters & hover)
    'Bicicletta elettrica': 2,
    'Quadriciclo leggero': 2,
    'Quadriciclo': 2,
    'Ciclomotore': 2,
    'Ciclomotore con passeggero': 2,

    # Group 3 – Light motorised two- & three-wheelers
    'Motociclo a solo': 3,
    'Motociclo con passeggero': 3,
    'Motociclo di polizia': 3,
    'Filobus': 3,

    # Group 4 – Cars & authority vehicles
    'Autovettura privata': 4,
    'Autovettura di polizia': 4,
    'Autovettura di soccorso': 4,
    'Autovettura pubblica': 4,
    'Autovettura con rimorchio': 4,
    'Autoambulanza in servizio': 4,
    'Autoambulanza': 4,

    # Group 5 – Light commercial vehicles & campers
    'Autocarro inferiore 35 q.': 5,
    'Autocaravan': 5,
    'Motocarrozzetta': 5,
    'Motocarro': 5,

    # Group 6 – Medium-duty trucks & vans
    'Autocarro superiore 35 q.': 6,
    'Autotreno': 6,
    'Autogru': 6,
    'Autosnodato': 6,
    'Macchina operatrice': 6,
    'Rimorchio': 6,
    'Autoarticolato': 6,

    # Group 7 – Buses & coaches
    'Autobus di linea': 7,
    'Autobus urbano': 7,
    'Autobus turistico': 7,
    'Autobus in extraurbana': 7,

    # Group 8 – Heavy commercial & special vehicles
    'Veicolo speciale': 8,
    'Autoveicolo trasp.promisc.': 8,
    'Trattore stradale': 8,

    # Group 9 – Trains & trams
    'Treno': 9,
    'Vettura tranviaria': 9,

    # Group 0 - other and unknown (hit and run)
    'Altro': 0,
    'Unknown': 0
}).fillna(0)

# Optionally define group labels for clarity
group_labels = {
    0: 'Unknown',
    1: 'Pedal-powered',
    2: 'Micro-mobility',
    3: 'Light motorised 2/3-wheelers',
    4: 'Cars & authority vehicles',
    5: 'Light commercial',
    6: 'Medium-duty trucks/vans',
    7: 'Buses & coaches',
    8: 'Heavy commercial & special',
    9: 'Trains & trams'
}

In [77]:
analyse_column('vehicle_type')

,Count,% per category,Unique Accidents
vehicle_type,,,
4,13306,68.53,12229
3,3732,19.22,3480
5,859,4.42,807
0,637,3.28,600
2,287,1.48,269
7,284,1.46,273
1,144,0.74,140
9,76,0.39,73
6,68,0.35,61


We will create a new binary feature, vehicle_unknown, for the accidents where the vehicle was described as 'other' or unknown (the driver did not stop). The remaining rows will use number 1 to 9 for the ordered categorical feature, vehicle_type.

In [78]:
# We create a binary flag for unknown vehicles
df["vehicle_unknown"] = (df["vehicle_type"] == 0).astype("int8")

# Replace 0 with NaN so it doesn’t interfere with the ordering
df.loc[df["vehicle_type"] == 0, "vehicle_type"] = pd.NA

# Make vehicle_type an ordered categorical (1–9, light → heavy)
df["vehicle_type"] = pd.Categorical(
    df["vehicle_type"],
    categories=[1, 2, 3, 4, 5, 6, 7, 8, 9],  # increasing weight
    ordered=True
)

#### (K) TIPOVEICOLO COLUMN - ENCODING COMPLETE

    NEW BINARY FEATURE: vehicle_unknown
    
    ORDERED CATEGORICAL FEATURE: vehicle_type
    1: 'Pedal-powered',
    2: 'Micro-mobility',
    3: 'Light motorised 2/3-wheelers',
    4: 'Cars & authority vehicles',
    5: 'Light commercial',
    6: 'Medium-duty trucks/vans',
    7: 'Buses & coaches',
    8: 'Heavy commercial & special',
    9: 'Trains & trams'

### STATOVEICOLO COLUMN

In [79]:
analyse_column('StatoVeicolo')

,Count,% per category,Unique Accidents
StatoVeicolo,,,
In marcia / fermata / arresto,18484,95.2,17078
Allontanatosi,932,4.8,876


In [80]:
df['hit_and_run'] = df['StatoVeicolo'].map({'Allontanatosi': 1,
                                            'Sosta': 0,
                                            'In marcia / fermata / arresto': 0}).fillna(0)

In [81]:
analyse_column('hit_and_run')

,Count,% per category,Unique Accidents
hit_and_run,,,
0,18484,95.2,17078
1,932,4.8,876


In [82]:
df['hit_and_run'] = pd.Categorical(
    df['hit_and_run'],
    categories=[0, 1],  # increasing weight
    ordered=False
)

### CLEAN UP COLUMNS

In [83]:
df.columns

Index(['Protocollo', 'TipoVeicolo', 'StatoVeicolo', 'DataOraIncidente',
       'CondizioneAtmosferica', 'Traffico', 'Gruppo', 'Localizzazione1',
       'STRADA1', 'Localizzazione2', 'STRADA2', 'Strada02', 'Chilometrica',
       'DaSpecificare', 'Latitude', 'Longitude', 'TipoStrada', 'Illuminazione',
       'parked_vehicle_involved', 'driver_injury', 'driver_gender',
       'passenger_no_of_females', 'passenger_no_of_males',
       'passenger_no_of_unknown_sex', 'passengers_killed',
       'passengers_uninjured', 'passengers_injured', 'phase_of_day', 'YEAR',
       'MONTH', 'DATE', 'TIME', 'DAY', 'pedestrian_gender', 'road_conditions',
       'road_markings_absent', 'road_markings_onroad',
       'road_markings_signposts', 'road_markings_temporary', 'road_features',
       'road_markings_traffic_lights', 'road_surface', 'accident_type',
       'TipoStradaDifficulty', 'visibility', 'Survival', 'pedestrian_injury',
       'lighting_insufficient', 'traffic_density', 'vehicle_type',
       

In [84]:
df.drop(columns=['StatoVeicolo', 'TipoVeicolo', 'TipoStrada',
        'Illuminazione', 'Traffico', 'Survival'], inplace=True)

In [85]:
df.columns

Index(['Protocollo', 'DataOraIncidente', 'CondizioneAtmosferica', 'Gruppo',
       'Localizzazione1', 'STRADA1', 'Localizzazione2', 'STRADA2', 'Strada02',
       'Chilometrica', 'DaSpecificare', 'Latitude', 'Longitude',
       'parked_vehicle_involved', 'driver_injury', 'driver_gender',
       'passenger_no_of_females', 'passenger_no_of_males',
       'passenger_no_of_unknown_sex', 'passengers_killed',
       'passengers_uninjured', 'passengers_injured', 'phase_of_day', 'YEAR',
       'MONTH', 'DATE', 'TIME', 'DAY', 'pedestrian_gender', 'road_conditions',
       'road_markings_absent', 'road_markings_onroad',
       'road_markings_signposts', 'road_markings_temporary', 'road_features',
       'road_markings_traffic_lights', 'road_surface', 'accident_type',
       'TipoStradaDifficulty', 'visibility', 'pedestrian_injury',
       'lighting_insufficient', 'traffic_density', 'vehicle_type',
       'vehicle_unknown', 'hit_and_run'],
      dtype='object')

In [86]:
df.to_parquet('008_data_only_pedestrians.parquet', index=False)

In [ ]:
metadata = {
    'notebook': '008 Data cleaning encoding ordered features.ipynb',
    'step': 'set ordered categorical encodings + create day-group features',

    # IO lineage
    'initial_parquet_file': '007_data_only_pedestrians.parquet',
    'final_parquet_file': '008_data_encoding_ordered.parquet',

    # Shapes (ordered encodings + small feature adds; no row filters)
    'initial_rows': 19_416,
    'initial_columns': 49,
    'final_rows': 19_416,
    'final_columns': 58,  # +9 new columns listed below

    # Columns
    'columns_added': [
        # ordinal code columns (kept originals as ordered categoricals)
        'visibility_ord', 'road_conditions_ord', 'road_surface_ord',
        'phase_of_day_ord', 'traffic_density_ord',
        # day-group features
        'day_weekend_binary', 'day_3way', 'day_4way', 'day_pairs'
    ],
    'columns_removed': [],
    'columns_set_ordered': [
        # originals that now have ordered pandas CategoricalDtype
        'Visibilita', 'road_conditions', 'road_surface', 'phase_of_day', 'traffic_density'
    ],

    # Explicit ordinal mappings (low→high risk/intensity)
    'ordinal_mappings': {
        'visibility_ord': {'Insufficiente': 0, 'Sufficiente': 1, 'Buona': 2},
        'road_conditions_ord': {'dry': 0, 'wet': 1, 'slippery': 2, 'icy': 3, 'road_conditions_unknown': None},
        'road_surface_ord': {'Tarmac': 0, 'Paved': 1, 'Damaged': 2, 'Graveled': 3, 'road_surface_unknown': None},
        'phase_of_day_ord': {'day': 0, 'twilight': 1, 'night': 2},
        # Accept either English or Italian labels; unknowns → None
        'traffic_density_ord': {'low': 0, 'moderate': 1, 'medium': 1, 'heavy': 2, 'congested': 3,
                                'scarso': 0, 'normale': 1, 'intenso': 2, 'congestionato': 3}
    },

    # Day-group feature definitions (as discussed earlier)
    'day_group_definitions': {
        'day_weekend_binary': {0: 'Weekday (Mon–Fri)', 1: 'Weekend (Sat–Sun)'},
        'day_3way': {0: 'Midweek (Tue–Thu)', 1: 'Friday', 2: 'Weekend (Sat–Sun)'},
        'day_4way': {0: 'Midweek (Tue–Thu)', 1: 'Monday', 2: 'Friday', 3: 'Weekend (Sat–Sun)'},
        'day_pairs': {0: 'Monday', 1: 'Tue+Wed', 2: 'Thu+Fri', 3: 'Sat+Sun'}
    },

    # QA checks
    'qa_checks': {
        'ordered_dtypes_set': True,   # originals now ordered categoricals
        'na_after_encoding': {'visibility_ord': 0, 'road_conditions_ord': '<fill>', 'road_surface_ord': '<fill>',
                              'phase_of_day_ord': 0, 'traffic_density_ord': '<fill>'},
        'codes_in_expected_range': True,   # all integer codes within mapping domain
        'row_count_unchanged': True
    },

    # Decisions & rationale
    'decisions_made': [
        'Kept original categorical columns as ordered pandas Categoricals for readability and profiling.',
        'Added *_ord numeric codes only for algorithms that benefit from ordinal distances; unknowns left as NaN.',
        'Defined day-group features (binary/3-way/4-way/pairs) to support ablation tests and interpretable clustering.',
        'Adopted risk-consistent orderings (e.g., dry<wet<slippery<icy; day<twilight<night).'
    ],

    # Notes for modeling
    'modeling_notes': (
        "For k-prototypes, prefer the ordered categoricals over *_ord codes to avoid overweighting numeric distances; "
        "use *_ord only in ablation or algorithms expecting numeric order. Re-tune gamma after feature changes."
    )
}

# Optional: persist as JSON next to outputs
# import json, pandas as pd
# metadata['run_timestamp'] = pd.Timestamp.now(tz='Europe/Rome').isoformat()
# with open('008_data_encoding_ordered_metadata.json', 'w', encoding='utf-8') as f:
#     json.dump(metadata, f, ensure_ascii=False, indent=2)